In [6]:
!pip install transformers torch pdfplumber pytesseract pillow matplotlib PyPDF2 python-igraph cairocffi gradio

In [7]:
import gradio as gr
import pdfplumber
import pytesseract
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import igraph as ig
import matplotlib.pyplot as plt
import pandas as pd
from PyPDF2 import PdfReader
import io
import os
import tempfile
import warnings
import sys

# Suppress Hugging Face warnings about long sequences (handled by chunking)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- GLOBAL MODEL/PIPELINE INITIALIZATION ---
# This is done once when the app starts for efficiency
MODEL_NAME = "CyberPeace-Institute/SecureBERT-NER"
# Use a simple flag to track successful initialization
MODEL_INITIALIZED = False
tokenizer = None
ner_pipeline = None

try:
    print("Attempting to load SecureBERT-NER Model...")
    global tokenizer, model, ner_pipeline
    warnings.warn("If the app hangs, the NER model is downloading or initializing. This may take a moment.")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
    ner_pipeline = pipeline(
        "token-classification",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="simple"
    )
    print("Model loaded successfully.")
    MODEL_INITIALIZED = True
except Exception as e:
    # Print detailed error to the console for debugging the environment
    print(f"CRITICAL ERROR: Failed to load model or tokenizer. CTI functionality will be disabled.")
    print(f"Details: {e}")
    # Ensure the error is explicitly propagated if processing is attempted

# Global variable to store the generated graph and entities
GLOBAL_GRAPH = None
GLOBAL_ENTITIES_DF = None

# --- CORE UTILITY FUNCTIONS ---

def extract_pdf_text(pdf_path):
    """Extracts text from all pages of a PDF."""
    try:
        # Use PdfReader, which handles file paths from Gradio's temp system
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            # Added space to prevent word joining during extraction
            extracted = page.extract_text()
            if extracted:
                text += extracted + " \n"
        return text
    except Exception as e:
        # Capture and report error details clearly
        return f"Error reading PDF file: {type(e).__name__}: {str(e)}"

def chunk_text(text, max_length=512, overlap=50):
    """Tokenizes text and creates overlapping chunks for NER processing."""
    if not MODEL_INITIALIZED:
        return ["Model not loaded."]

    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(tokens), max_length - overlap):
        chunk = tokens[i:i + max_length]
        chunks.append(tokenizer.decode(chunk))
    return chunks

# --- IGRAPH CONSTRUCTION LOGIC (Corrected and Robust) ---

def build_cti_knowledge_graph_igraph(entities, labels):
    """
    Constructs an iGraph graph, filtering out empty entities
    and using custom CTI rules for edge labeling.
    """
    name_to_original_label = {}
    vertex_names = []

    # 1. Collect unique, cleaned vertices and map them to their labels
    for ent, lab in zip(entities, labels):
        clean_ent = ent.replace('\n', ' ').strip()

        # CRITICAL FILTER: Skip if the entity is an empty string
        if clean_ent and clean_ent not in name_to_original_label:
            name_to_original_label[clean_ent] = lab
            vertex_names.append(clean_ent)

    # 2. Initialize the graph
    G = ig.Graph(directed=True)
    G.add_vertices(len(vertex_names))
    G.vs["name"] = vertex_names

    # Map attributes
    G.vs["node_type"] = [name_to_original_label[name] for name in G.vs["name"]]
    G.vs["label"] = G.vs["name"]
    # Use distinct colors based on node type
    color_map = {
        'ACT': '#1f78b4', # Blue for Actions
        'TOOL': '#33a02c', # Green for Tools
        'IDTY': '#ff7f00', # Orange for Identities
        'TIME': '#cab2d6', # Purple for Time
        'MISC': '#a6cee3', # Light Blue for Misc
        'APT': '#e31a1c', # Red for APTs
        'VULID': '#ffff99', # Yellow for Vulnerabilities
        'IP': '#fdbf6f', # Light orange for IPs
        'URL': '#ff7f00', # Orange for URLs
        'DOMAIN': '#b2df8a', # Light green for Domains
        'FILE': '#fb9a99', # Pink for Files
        'HASH': '#a6cee3', # Light blue for Hashes
        'CVE': '#ffff99', # Yellow for CVEs (alias for VULID)
        'OS': '#cab2d6', # Purple for OS
        'PROTOCOL': '#fdbf6f' # Light orange for Protocols
    }
    G.vs["color"] = [color_map.get(lab, '#a6cee3') for lab in G.vs["node_type"]]

    # 3. Add edges
    edges_to_add = []
    edge_relations = []
    cleaned_entities = [ent.replace('\n', ' ').strip() for ent in entities]
    cleaned_labels = labels # Keep original labels aligned with cleaned entities

    for i in range(len(cleaned_entities) - 1):
        e1, l1 = cleaned_entities[i], cleaned_labels[i]
        e2, l2 = cleaned_entities[i+1], cleaned_labels[i+1]

        # Skip if either entity was filtered out
        if not e1 or not e2:
            continue

        # Check if entities exist in the final vertex list (they should, but double-check)
        if e1 not in G.vs["name"] or e2 not in G.vs["name"]:
            continue

        # Get vertex IDs for the edge
        id1 = G.vs.find(name=e1).index
        id2 = G.vs.find(name=e2).index

        relation = ""
        # --- CTI RULES ---
        if l1 == "IDTY" and l2 == "ACT":
            relation = "performs_ttp"
        elif l1 == "ACT" and l2 == "TOOL":
            relation = "targets_platform"
        elif l1 == "ACT" and l2 == "IDTY":
            relation = "used_by_tactic"
        elif l1 == "TOOL" and l2 == "ACT":
            relation = "targeted_by"
        elif l1 == "TOOL" and l2 == "TOOL":
            relation = "involves_tech"
        elif l1 == "APT" and l2 == "MALWARE":
             relation = "uses"
        elif l1 == "MALWARE" and l2 in ["IP", "URL", "DOMAIN", "FILE", "HASH"]:
             relation = "uses_indicator"
        elif l1 in ["IP", "URL", "DOMAIN", "FILE", "HASH"] and l2 in ["MALWARE", "TOOL"]:
             relation = "indicates"
        elif l1 == "VULID" and l2 in ["OS", "TOOL"]:
             relation = "affects"
        elif l1 in ["OS", "TOOL"] and l2 == "VULID":
             relation = "vulnerable_to"
        elif l1 == "TIME" and l2 in ["ACT", "IDTY", "TOOL", "APT"]: # Expanded time rule
            relation = "observed_during"
        else:
            relation = "related_to"


        edges_to_add.append((id1, id2))
        edge_relations.append(relation)

    G.add_edges(edges_to_add)
    G.es["label"] = edge_relations
    G.es["color"] = "gray"

    return G

# --- VISUALIZATION FUNCTION ---

def query_entity_graph_igraph(G, entity_name):
    """
    Generates a 1-hop subgraph plot for the selected entity.
    """
    clean_name = entity_name.replace('\n', ' ').strip()

    if clean_name not in G.vs["name"]:
        # Use None to clear the plot if the entity isn't found
        return None, f"Entity '{clean_name}' not found or has no connections."

    try:
        center_vid = G.vs.find(name=clean_name).index
        # Get 1-hop neighborhood
        neighbor_vids = G.neighbors(center_vid, mode="all")
        subgraph_vids = list(set([center_vid] + neighbor_vids))

        # Create the induced subgraph
        subgraph = G.induced_subgraph(subgraph_vids)

        # Layout and plotting logic
        layout = subgraph.layout("kamada_kawai")

        visual_style = {}
        visual_style["vertex_label"] = subgraph.vs["name"]
        visual_style["vertex_color"] = subgraph.vs["color"]
        visual_style["edge_label"] = subgraph.es["label"]
        visual_style["edge_color"] = "gray"
        visual_style["vertex_size"] = 25
        visual_style["vertex_label_size"] = 10
        visual_style["edge_label_size"] = 9
        visual_style["bbox"] = (800, 600)
        visual_style["margin"] = 50

        fig, ax = plt.subplots(figsize=(10, 8))
        ig.plot(subgraph, target=ax, layout=layout, **visual_style)

        ax.set_title(f"Knowledge Graph: 1-Hop Neighbors of '{clean_name}'", fontsize=14)

        # Must explicitly call show() to ensure display in some environments (though Gradio handles figure objects)
        # However, for safety, just return the figure object for Gradio to handle the rest.
        return fig, f"Successfully mapped {subgraph.vcount()} connections for '{clean_name}'."

    except Exception as e:
        # Close figure in case an error occurred after opening it
        plt.close(fig)
        return None, f"Error generating subgraph: {type(e).__name__}: {str(e)}"


# --- GRADIO INTERFACE LOGIC ---

def process_cti_report(file_obj):
    """
    The main processing function: extracts text, runs NER, builds the graph,
    and updates the Gradio components.
    """
    global GLOBAL_GRAPH, GLOBAL_ENTITIES_DF

    # Define helper for resetting outputs
    initial_dropdown_choices = []
    initial_dropdown_value = None
    initial_df = pd.DataFrame(columns=['Entity', 'Type', 'Score'])
    initial_graph = None
    initial_graph_status = ""
    initial_status = ""


    if file_obj is None:
        return initial_df, gr.Dropdown(choices=initial_dropdown_choices, value=initial_dropdown_value), initial_graph, initial_graph_status, "Please upload a PDF file."

    # Check 1: Model Initialization
    if not MODEL_INITIALIZED:
        return initial_df, gr.Dropdown(choices=initial_dropdown_choices, value=initial_dropdown_value), initial_graph, initial_graph_status, "CRITICAL: NER Model failed to load during startup. Cannot process report."

    pdf_path = file_obj.name

    # 1. Extract Text
    text = extract_pdf_text(pdf_path)
    if text.startswith("Error reading PDF"):
        # Returns the detailed error message from extract_pdf_text
        return initial_df, gr.Dropdown(choices=initial_dropdown_choices, value=initial_dropdown_value), initial_graph, initial_graph_status, text

    # 2. Chunk Text and Run NER
    chunks = chunk_text(text)

    results = []
    try:
        for chunk in chunks:
            res = ner_pipeline(chunk)
            results.extend(res)
    except Exception as e:
        return initial_df, gr.Dropdown(choices=initial_dropdown_choices, value=initial_dropdown_value), initial_graph, initial_graph_status, f"NER Pipeline runtime error: {type(e).__name__}: {str(e)}"


    if not results:
        return initial_df, gr.Dropdown(choices=initial_dropdown_choices, value=initial_dropdown_value), initial_graph, initial_graph_status, "NER returned no entities. Report may be too complex or model failed quietly."

    # 3. Create DataFrame and Entity/Label lists
    df = pd.DataFrame(results)

    # Clean up DataFrame for display (rounding scores, simplifying columns)
    df = df.rename(columns={'word': 'Entity', 'entity_group': 'Type'})
    df['Score'] = df['score'].round(4)
    df_display = df[['Entity', 'Type', 'Score']].copy()

    # Prepare lists for graph building
    entities = df["Entity"].tolist() # Use the cleaned 'Entity' column
    labels = df["Type"].tolist()

    # 4. Build iGraph Knowledge Graph
    try:
        G = build_cti_knowledge_graph_igraph(entities, labels)
    except Exception as e:
        return df_display, gr.Dropdown(choices=initial_dropdown_choices, value=initial_dropdown_value), initial_graph, initial_graph_status, f"Error building graph: {type(e).__name__}: {str(e)}"

    # 5. Store global variables and prepare dropdown
    GLOBAL_GRAPH = G
    GLOBAL_ENTITIES_DF = df_display

    # Get the list of unique entity names for the dropdown
    unique_entity_names = G.vs["name"]

    final_status = f"Successfully processed report. Extracted {G.vcount()} unique entities, built graph with {G.ecount()} edges."
    # Return the dropdown choices and initial value (None) along with other outputs
    return df_display, gr.Dropdown(choices=unique_entity_names, value=None), initial_graph, initial_graph_status, final_status


def update_subgraph(entity_name):
    """
    Handles the dropdown selection and plots the subgraph.
    """
    if GLOBAL_GRAPH is None or entity_name is None:
        return None, "Please process a report first and select an entity."

    # We pass the global graph to the plotting function
    return query_entity_graph_igraph(GLOBAL_GRAPH, entity_name)


# --- GRADIO INTERFACE LAYOUT ---

with gr.Blocks(title="CTI Knowledge Graph Builder") as app:
    gr.Markdown("# Cyber Threat Intelligence (CTI) Knowledge Graph Analyzer")
    gr.Markdown("Upload a CTI report (PDF) to extract entities, build the knowledge graph, and visualize 1-hop attack chains.")

    with gr.Row():
        file_input = gr.File(label="Upload CTI Report (PDF)", file_types=[".pdf"])
        process_button = gr.Button("Process Report", variant="primary")
        status_output = gr.Textbox(label="Status", interactive=False)

    # --- Section 1: Extracted Entities ---
    gr.Markdown("---")
    gr.Markdown("## Step 1: Extracted Entities (NER Results)")
    entity_table_output = gr.DataFrame(
        headers=["Entity", "Type", "Score"],
        col_count=(3, "fixed"),
        label="Extracted Entities (Words, Types, Confidence)",
        interactive=False,
    )

    # --- Section 2: Interactive Graph Query ---
    gr.Markdown("---")
    gr.Markdown("## Step 2: Visualize Knowledge Graph Subgraph")

    with gr.Row():
        entity_dropdown = gr.Dropdown(
            label="Select an Entity (Node) to Query",
            choices=[], # Initial empty
            interactive=True,
            scale=2
        )
        query_button = gr.Button("Show Subgraph", scale=1)

    graph_output = gr.Plot(label="1-Hop Knowledge Graph Subgraph", visible=True)
    graph_status = gr.Textbox(label="Graph Status", interactive=False)

    # --- EVENT HANDLERS ---

    # 1. Handle file upload and initial processing
    process_button.click(
        fn=process_cti_report,
        inputs=[file_input],
        outputs=[entity_table_output, entity_dropdown, graph_output, graph_status, status_output] # Corrected outputs list
    )

    # 2. Handle entity selection and subgraph generation
    query_button.click(
        fn=update_subgraph,
        inputs=[entity_dropdown],
        outputs=[graph_output, graph_status]
    )

    entity_dropdown.select(
        fn=update_subgraph,
        inputs=[entity_dropdown],
        outputs=[graph_output, graph_status]
    )

app.launch()

Attempting to load SecureBERT-NER Model...


Device set to use cpu


Model loaded successfully.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d61f7a7dd9792ce00.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
